# Data Spilling

This notebook creates the train test splits that are required for modelling

In [ ]:
import sys
sys.path.append('../src')

In [ ]:
import os
from pprint import pprint
import pickle
import pandas as pd
import numpy as np
from keras.utils import timeseries_dataset_from_array
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from util_IO import (
    load_pickle_from_main_project_dir,
    load_attributes_df,
    load_timeseries_df
)
from datetime import datetime
from tqdm import tqdm

# Set pandas to display a maximum of 300 columns
pd.set_option('display.max_columns', 300)
pd.set_option('display.max_rows', 1000)

# Suppress the SettingWithCopyWarning
pd.options.mode.chained_assignment = None

## Configuration

In [ ]:
# Validation run?
VALIDATION = False

# Use existing train test split for the local model in "models" folder?
USE_EXISTING_TRAIN_TEST_SPLIT_FOR_LOCAL_MODEL = True

# Setting 
EXAMPLE_INPUT = False

# Number of bins in stratification
number_of_bins = 5

### Load metadata from *1-DataAggregation.ipynb*

In [ ]:
aggr_parameters_dict, camels_gb_use_case_dir = load_pickle_from_main_project_dir(
    'aggr_parameters_dict.pkl'
)

In [ ]:
# Variables picked
attributes_index = aggr_parameters_dict["attributes"]["attributes_index"]
date_field = aggr_parameters_dict['timeseries']['date_field']
label_field = aggr_parameters_dict['timeseries']['label_field']
camels_gb_silver_dir = aggr_parameters_dict['camels_gb_silver_dir']
camels_gb_data_attributes_aggr_dir = aggr_parameters_dict['camels_gb_data_attributes_aggr_dir']
camels_gb_data_timeseries_aggr_dir = aggr_parameters_dict['camels_gb_data_timeseries_aggr_dir']

### Define reference data

In [ ]:
if EXAMPLE_INPUT:

    # Example 2D dataset with date, dynamic, and label columns
    timeseries = {
        date_field: pd.date_range(start='2023-01-01', periods=20+15+10, freq='D'),
        'catchmentID': ['10002'] * 20 + ['10003'] * 15 + ['10004'] * 10,
        f"{date_field}_group": ['00'] * 15 + ['01'] * 5 + ['0'] * 8 + ['01'] * 7 + ['00'] * 8 + ['01'] * 2,
        'precipitation': np.random.rand(20+15+10)*100 + 100,
        'temperature': np.random.rand(20+15+10)*100 + 1000,
        'humidity': np.random.rand(20+15+10)*100 + 10000,
        'time_ref': range(20+15+10),
        label_field: np.arange(20+15+10)*20
    }

    # Static variables
    attributes = {
        'catchmentID': ['10002', '10003', '10004'],
        'silt_perc': [.1, .2, .3],
        'clay_perc': [.5, .6, .7],
        'var_1': [15, 20, 25],
        'var_2': [150, 200, 250],
    }

    # Create DataFrames
    attributes_df = pd.DataFrame(attributes).set_index('catchmentID')
    timeseries_df = pd.DataFrame(timeseries)

    # __________
    # Timeseries
    
    # Variables to be windowed
    X_vars_names = [
        "precipitation",
        "temperature",
        "humidity",
        "time_ref"
    ]
    
    # Define which columns need to be scaled with specific scaler
    X_minmax_columns_names_list = ['time_ref']  # Column names for MinMaxScaler
    X_standard_columns_names_list = ['precipitation', 'temperature', 'humidity']  # Column names for StandardScaler
    
    # Get sets to perform checks and warnings
    X_minmax_columns_names_set = set(X_minmax_columns_names_list)
    X_standard_columns_names_set = set(X_standard_columns_names_list)

    # Define window size
    sequence_length = 2

    # Define label field for model
    label_field_feed = label_field
    
    # Time range's first year
    start_year = 2020


    # __________
    # Attributes
    
    X_static_vars_names = attributes_df.columns.to_list()
    X_static_minmax_columns_names_list = ['silt_perc']
    X_static_standard_columns_names_list = ['var_1', 'var_2']

    X_static_minmax_columns_names_set = set(X_static_minmax_columns_names_list)
    X_static_standard_columns_names_set = set(X_static_standard_columns_names_list)


    # _______________________________
    # Attributes & Timeseries lookups
    
    # "Y", so it does not trigger sample reduction
    cs = "Y"
    
else:

    # ____________
    # Loading data

    # Timeseries
    
    timeseries_df = load_timeseries_df(
        camels_gb_data_timeseries_aggr_dir,
        "timeseries_postFEa.csv",
        date_field
    )

    display(timeseries_df.head(3))

    
    # Attributes  

    attributes_df = load_attributes_df(
        camels_gb_data_attributes_aggr_dir,
        "fundamental_postFEa.csv",
        attributes_index
    )

    display(attributes_df.head(3))
    
    
    # ___________________
    # Time series lookups
    
    # Variables to be windowed
    X_vars_names = [
        "precipitation",
        "temperature",
        "humidity",
        "shortwave_rad",
        "longwave_rad",
        "windspeed",
        "sin_year",
        "cos_year",
        "time_ref"
    ]
    
    # Column names for MinMaxScaler
    X_minmax_columns_names_list = [     
        'time_ref'
    ] 
    
    # Column names for StandardScaler
    X_standard_columns_names_list = [
        "precipitation",
        "temperature",
        "humidity",
        "shortwave_rad",
        "longwave_rad",
        "windspeed"
    ]  
    
    # Get sets to perform checks and warnings
    X_minmax_columns_names_set = set(X_minmax_columns_names_list)
    X_standard_columns_names_set = set(X_standard_columns_names_list)

    # Define window size
    sequence_length = 30 # 30 15 10 5
    
    # Define transformed label - ALREADY PRESENT IN TIME SERIES DATASET
    label_transformation = "log1p"
    
    # DERIVE transformed label field name - ALREADY PRESENT IN TIME SERIES DATASET
    label_transformed_field = f"{label_transformation}_{label_field}"

    # Which one do you want??
    label_field_feed = label_field # label_transformed_field / label_field
    
    # Time range's first year
    start_year = 1985

    
    # __________________
    # Attributes lookups

    X_static_vars_names = attributes_df.columns.to_list()
    X_static_minmax_columns_names_list = []
    X_static_standard_columns_names_list = [
        x for x in X_static_vars_names if x not in ['baseflow_index', 'chalk_stream_flag']
    ]
    
    X_static_minmax_columns_names_set = set(X_static_minmax_columns_names_list)
    X_static_standard_columns_names_set = set(X_static_standard_columns_names_list)


    # _______________________________
    # Attributes & Timeseries lookups

    # Do you want to include chalk stream catchments?
    cs = "Y" # "Y" / "N"

#### Checks & Warnings

In [ ]:
def validate_scaling(minmax_set, standard_set, all_vars, dataset_name):
    
    # Check for overlapping columns
    assert minmax_set.isdisjoint(standard_set), (
        f"""- {dataset_name} dataset -
There are repetitions between columns to be scaled with MinMax scaler and those to be scaled with Standard scaler.
Please ensure each column is assigned to only one scaler type.\n\n"""
    )

    # Identify and print unscaled columns
    no_scaled = list(set(all_vars) - minmax_set - standard_set)
    if no_scaled:
        print(f"""\n- {dataset_name.upper()} dataset -
Columns that will NOT be scaled:
""")
        pprint(no_scaled)

    return no_scaled


# TIME SERIES dataset
X_no_scaled_columns_names_list = (
    validate_scaling(
        X_minmax_columns_names_set,
        X_standard_columns_names_set,
        X_vars_names,
        "TIME SERIES"
    )
)

# ATTRIBUTES dataset
X_static_no_scaled_columns_names_list = (
    validate_scaling(
        X_static_minmax_columns_names_set,
        X_static_standard_columns_names_set,
        X_static_vars_names,
        "ATTRIBUTES"
    )
)

In [ ]:
if EXAMPLE_INPUT:
    display(attributes_df)

In [ ]:
if EXAMPLE_INPUT:
    display(timeseries_df)

### General parameters

In [ ]:
train_size = 0.7
test_size = 1 - train_size
min_required_obs = sequence_length / test_size
X_n_vars=len(X_vars_names)

# Attributes

## Chalk streams

In [ ]:
# If chalk stream catchments need to be removed
if cs == "N":

    # Store "pre" situation
    n_row_pre = attributes_df.shape[0]
    
    # Define chalk stream catchments list [🚩 useful for time series too]
    chalk_streams_list = attributes_df[attributes_df['chalk_stream_flag'] == True].index.to_list()
    
    # Remove chalk stream catchments
    attributes_df = attributes_df[~attributes_df.index.isin(chalk_streams_list)]

    # Remove chalk stream flag columns
    (
        attributes_df
            .drop(
                columns='chalk_stream_flag',
                inplace=True
        )
    )

    assert n_row_pre > attributes_df.shape[0], ('Inconsistency after trying to remove chalk stream catchments')

    
    # Remove 'chalk_stream_flag' from name lists
    # 🚩 Although just one list will have 'chalk_stream_flag', it is precautionarily removed from all the registy lists
    X_static_vars_names = [
        x for x in X_static_vars_names if x != 'chalk_stream_flag'
    ]
    X_static_minmax_columns_names_list = [
        x for x in X_static_minmax_columns_names_list if x != 'chalk_stream_flag'
    ]
    X_static_standard_columns_names_list = [
        x for x in X_static_standard_columns_names_list if x != 'chalk_stream_flag'
    ]
    X_static_no_scaled_columns_names_list = [
        x for x in X_static_no_scaled_columns_names_list if x != 'chalk_stream_flag'
    ]

# Timeseries

## Filter for time range

In [ ]:
timeseries_df = (
    timeseries_df[
        pd.to_datetime(timeseries_df[date_field]) >= datetime(start_year, 9, 30) # Because most recent observation: 30/09/2015
    ]
)

## Chalk streams

In [ ]:
if cs == "N":

    # Store "pre" situation
    n_row_pre = timeseries_df.shape[0]
    
    timeseries_df = (
        timeseries_df[
            ~timeseries_df['catchmentID'].isin(chalk_streams_list)
        ]
    )

    assert n_row_pre > timeseries_df.shape[0], ("Inconsistency after trying to remove chalk stream catchments' TS")

## Sensor data windowing and registry (function definition)

## Sensor data aggregation

In [ ]:
def create_timeseries_for_sensor(
    sensor_data,
    sensor_id,
    group,
    sequence_length=15,
    sampling_rate=1,
    sequence_stride=1
):
    
    # ____________________________________
    # Split train/test (without shuffling)
    
    # Train and test set
    X_train, X_test, y_train, y_test = (
        train_test_split(
            sensor_data[X_vars_names].values,
            sensor_data[label_field_feed].values,
            train_size=train_size,
            shuffle=False
        )
    )
    
    # Timestamps
    timestamps_train, timestamps_test = (
        train_test_split(
            sensor_data[date_field].values,
            train_size=train_size,
            shuffle=False
        )
    )
    
    
    # _________
    # Windowing
    for X, y, timestamps, curr_set in zip(
        [X_train, X_test],
        [y_train, y_test],
        [timestamps_train, timestamps_test],
        ['train', 'test']
    ):
    
        # ______________________________
        # Windowing and numpy conversion

        # Launch `keras.utils.timeseries_dataset_from_array` to generate the windows
        dataset = timeseries_dataset_from_array(
            X,
            targets=y[(sequence_length-1):],
            sequence_length=sequence_length,
            sampling_rate=sampling_rate,
            sequence_stride=sequence_stride,
            batch_size=1 # Must be ️1, see related comment🅰️
        )

        # Collect the data from batches in the dataset
        sequence_list=[]
        target_list = []
        for curr_sequence, curr_target in dataset:  # See previous comment 🅰️
            sequence_list.append(curr_sequence)
            target_list.append(curr_target)

        # Convert into a 3D numpy array
        sequence = np.array(sequence_list).reshape(-1, sequence_length, X_n_vars)
        target = np.array(target_list).reshape(-1)


        # ____________________
        # Observation registry

        # Calculate timestamps tuple with start and end dates for each observation
        timestamps = [
            (timestamps[i], timestamps[i + sequence_length - 1])
            for i in range(len(timestamps) - sequence_length + 1)
        ]

        # Generate data frame to store registry
        sensor_registry_df = pd.DataFrame(timestamps, columns=['start_date', 'end_date'])
        sensor_registry_df.insert(0, 'group', group)
        sensor_registry_df.insert(0, 'catchmentID', sensor_id)


        # ______
        # Checks
        n_sequence = sequence.shape[0]
        n_target = target.shape[0]
        n_timestamps = len(timestamps)
        assert n_sequence == n_target == n_timestamps, "Sensor' sequence, target, and timestamp for data frame must have the same length for first dimension"
    
        # ______________
        # Set allocation
        if curr_set == 'train':
            sequence_train = sequence
            target_train = target
            sensor_registry_train_df = sensor_registry_df
        
        elif curr_set == 'test':
            sequence_test = sequence
            target_test = target
            sensor_registry_test_df = sensor_registry_df
    
    return (sequence_train, target_train), (sequence_test, target_test), (sensor_registry_train_df, sensor_registry_test_df)

In [ ]:
# Create unique CatchmentID list

if USE_EXISTING_TRAIN_TEST_SPLIT_FOR_LOCAL_MODEL:
    train_catchments = np.array(pd.read_csv(f"../datasets/train_unique_catchment_ids.csv")["catchmentID"].astype(str))
    test_catchments = np.array(pd.read_csv(f"../datasets/test_unique_catchment_ids.csv")["catchmentID"].astype(str))
else:
    # Calculate MEAN and MEAN per catchmentID
    df = timeseries_df.groupby('catchmentID', as_index=False).agg({'discharge_vol': 'mean'})
    
    # Create 5 bins for stratification based on NSE values
    df['discharge_vol_bin'] = pd.qcut(df['discharge_vol'], q=number_of_bins, labels=False, duplicates='drop')
    
    # Split the data into train and test sets with stratification
    train_df, test_df = train_test_split(
        df,
        test_size=test_size,
        stratify=df['discharge_vol_bin'],
    )
    
    # Convert to lists
    train_catchments = train_df["catchmentID"].to_list()
    test_catchments = test_df["catchmentID"].to_list()

In [ ]:
# Initialize aggregate variables
X_train,  X_test = np.empty((0, sequence_length, X_n_vars)), np.empty((0, sequence_length, X_n_vars))
y_train, y_test = np.array([]), np.array([])

# Initialize data frames for registry
registry_train_df = pd.DataFrame()
registry_test_df = pd.DataFrame()

# Set verbosity
verbose=EXAMPLE_INPUT

# Loop on sensor data to aggregate the data 
for (curr_sensor_id, curr_date_group), curr_sensor_df in tqdm(
        timeseries_df.groupby(['catchmentID', f"{date_field}_group"]),
        desc="Processing catchmentID-groups"
):

    # Define current "sensor-group"
    curr_sensor_group = f"{curr_sensor_id}-{curr_date_group}"
    
    # Calculate number of observations
    n_curr_sensor_obs = curr_sensor_df.shape[0]
    
    # Check on the minimum of observation required
    if n_curr_sensor_obs >= min_required_obs:

        if verbose:
            print(f"{curr_sensor_group}... ", end="")
    
        # Function call
        (curr_X_train, curr_y_train), (curr_X_test, curr_y_test), (curr_registry_train_df, curr_registry_test_df)  = (
            create_timeseries_for_sensor(
                curr_sensor_df,
                curr_sensor_id,
                curr_date_group,
                sequence_length=sequence_length)
        )

        if curr_sensor_id in train_catchments:
            print(f"Train catchment:- {curr_sensor_id}")
            curr_X_train = np.concatenate((curr_X_train, curr_X_test), axis=0)
            curr_y_train = np.concatenate((curr_y_train, curr_y_test), axis=0)
            curr_X_test = np.empty((0, sequence_length, X_n_vars))
            curr_y_test = np.array([])
            
            curr_registry_train_df = pd.concat((curr_registry_train_df, curr_registry_test_df))
            curr_registry_test_df = pd.DataFrame()
        elif curr_sensor_id in test_catchments:
            print(f"Test catchment:- {curr_sensor_id}")
            curr_X_train = np.empty((0, sequence_length, X_n_vars))
            curr_y_train = np.array([])
            curr_X_test = np.concatenate((curr_X_train, curr_X_test), axis=0)
            curr_y_test = np.concatenate((curr_y_train, curr_y_test), axis=0)
            
            curr_registry_train_df = pd.DataFrame()
            curr_registry_test_df = pd.concat((curr_registry_train_df, curr_registry_test_df))
        else:
            print("Missing catchment")

        # Aggregate Train
        X_train = np.concatenate((X_train, curr_X_train), axis=0)
        y_train = np.concatenate((y_train, curr_y_train), axis=0)

        # Aggregate Test
        X_test = np.concatenate((X_test, curr_X_test), axis=0)
        y_test = np.concatenate((y_test, curr_y_test), axis=0)

        # Aggregate registry
        registry_train_df = pd.concat([registry_train_df, curr_registry_train_df], ignore_index=True)
        registry_test_df = pd.concat([registry_test_df, curr_registry_test_df], ignore_index=True)
        
        if verbose:
            print("OK!\t", end="")
        
    else:
        
        if verbose:
            print(f"{curr_sensor_group}: too short timeseries ({n_curr_sensor_obs} obs.)\t", end="")


# Checks
n_sequence = X_train.shape[0]
n_target = y_train.shape[0]
n_registry = registry_train_df.shape[0]
assert n_sequence == n_target == n_registry, "Train sequence, target must, and registry data frame must have the same length for first dimension"

n_sequence = X_test.shape[0]
n_target = y_test.shape[0]
n_registry = registry_test_df.shape[0]
assert n_sequence == n_target == n_registry, "Tests sequence, target must, and registry data frame must have the same length for first dimension"

In [ ]:
if EXAMPLE_INPUT:
    print(X_train)
else:
    print(f"X train dimensions: {X_train.shape}")

In [ ]:
if EXAMPLE_INPUT:
    print(y_train)
else:
    print(f"y train dimensions: {y_train.shape}")

In [ ]:
if EXAMPLE_INPUT:
    print(X_test)
else:
    print(f"X test dimensions: {X_test.shape}")

In [ ]:
if EXAMPLE_INPUT:
    print(y_test)
else:
    print(f"y test dimensions: {y_test.shape}")

## `X_train` and `X_test` scaling

### Columns management

In [ ]:
# Get the indices of the columns
X_minmax_columns_index = [X_vars_names.index(col) for col in X_minmax_columns_names_list]
X_standard_columns_index = [X_vars_names.index(col) for col in X_standard_columns_names_list]
X_no_scaled_columns_index = [X_vars_names.index(col) for col in X_no_scaled_columns_names_list]

### `multi_scaling` function

In [ ]:
def multi_scaling(
    X,
    preprocessor
):
 
    # Store original dimensions
    n_samples, n_timesteps, n_features = X.shape
    print(f"Input with dimensions:\t\t\t{X.shape}")

    # Reshape to 2D
    X_2D = X.reshape(-1, n_features)
    print(f"Reshaped to 2D with dimensions:\t\t{X_2D.shape}")

    # Scale the 2D
    X_scaled_2D = preprocessor.transform(X_2D)
    print(f"After transformation dimensions:\t{X_scaled_2D.shape}")

    # Reshape back to 3D
    X_scaled_3D = X_scaled_2D.reshape(n_samples, n_timesteps, n_features)
    print(f"Reshaped to 3D with dimensions:\t\t{X_scaled_3D.shape}")

    return X_scaled_3D

### Scaling

In [ ]:
# _______________________
# Define the preprocessor

# Reshape train set (used as mold data) to 2D
X_mold = X_train.reshape(-1, X_n_vars)
print(f"Mold created with dimensions:\t\t\t\t{X_mold.shape}")

# Remove repeated rows which comes with the windowing (avoiding to distort the distribution)
X_mold = np.unique(X_mold, axis=0)
print(f"Mold reduced by duplicated rows with dimensions:\t{X_mold.shape}", end="\n")

# Define the ColumnTransformer
X_preprocessor = ColumnTransformer(
    transformers=[
        ('minmax', MinMaxScaler(), X_minmax_columns_index),
        ('standard', StandardScaler(), X_standard_columns_index),
        ('no_scale', 'passthrough', X_no_scaled_columns_index)
    ]
)

# Numpy new columns names order
X_cols_names = (
    X_minmax_columns_names_list
    + X_standard_columns_names_list
    + X_no_scaled_columns_names_list
)

# Fit the preprocessor
X_preprocessor.fit(X_mold)

print("\nTrain")

# _____________
# Apply scaling

# Train
X_train_scaled = multi_scaling(
    X_train,
    X_preprocessor
)

print("\nTest")

# Test
X_test_scaled = multi_scaling(
    X_test,
    X_preprocessor
)

print("\nNew columns order:")
pprint(X_cols_names)

In [ ]:
if EXAMPLE_INPUT:
    print(X_train_scaled)
else:
    print(f"Scaled X train dimensions:{X_train_scaled.shape}")

In [ ]:
if EXAMPLE_INPUT:
    print(X_test_scaled)
else:
    print(f"Scaled X test dimensions: {X_test_scaled.shape}")

## Train set shuffling

In [ ]:
# Get the number of samples
n_samples = X_train_scaled.shape[0]

# Set the seed
np.random.seed(82)

# Generate a random permutation of indices
shuffle_indices = np.random.permutation(n_samples)

# Shuffle the X training set
X_train_scaled_shuffled = X_train_scaled[shuffle_indices]

# Shuffle the y training set
y_train_shuffled = y_train[shuffle_indices]

# Shuffle the dataframe using the same indices
# 🚩 `iloc` method: without resetting the index!!
registry_train_df_shuffled = registry_train_df.iloc[shuffle_indices]

In [ ]:
if EXAMPLE_INPUT:
    print(X_train_scaled_shuffled)
else:
    print(f"Shuffled & scaled X train dimensions: {X_train_scaled_shuffled.shape}")

In [ ]:
if EXAMPLE_INPUT:
    print(y_train_shuffled)
else:
    print(f"Shuffled & scaled y train dimensions: {y_train_shuffled.shape}")

In [ ]:
if EXAMPLE_INPUT:
    display(registry_train_df_shuffled)
else:
    print(f"Shuffled registry df dimensions: {registry_train_df_shuffled.shape}")

# Attributes (*static variables*)

## Attributes merge via registry (function definition)

In [ ]:
def merge_attributes(registry):
    
    # Define columns not to be put in the output
    # 🚩 any columns from the registry which is not 'catchmentID' 🚩
    registry_redundant_columns_set = set(registry.columns)
    registry_redundant_columns_set.remove('catchmentID')

    # Perform the left join
    merged_df = registry.merge(
        attributes_df,
        left_on='catchmentID',
        right_index=True,
        how='left'
    )

    # Drop redundant columns
    merged_df = merged_df.drop(columns=registry_redundant_columns_set)

    # Set the index to 'catchmentID'
    merged_df = merged_df.set_index('catchmentID')
    
    return merged_df

## Train and test set merge & scale

In [ ]:
# Get the indices of the columns for scalers
X_static_minmax_columns_index = [X_static_vars_names.index(col) for col in X_static_minmax_columns_names_list]
X_static_standard_columns_index = [X_static_vars_names.index(col) for col in X_static_standard_columns_names_list]
X_static_no_scaled_columns_index = [X_static_vars_names.index(col) for col in X_static_no_scaled_columns_names_list]


# _________________
# Scaler definition

# Define transformer
X_static_preprocessor = ColumnTransformer(
    transformers=[
        ('minmax', MinMaxScaler(), X_static_minmax_columns_index),
        ('standard', StandardScaler(), X_static_standard_columns_index),
        ('no_scale', 'passthrough', X_static_no_scaled_columns_index)
    ]
)

# (Re)Define columns order
X_static_cols_names = (
    X_static_minmax_columns_names_list
    + X_static_standard_columns_names_list
    + X_static_no_scaled_columns_names_list
)

# Define mold
X_static_preprocessor.fit(attributes_df)


# ________________________
# Transform sets TRAIN set

# Merge
X_train_static_df = merge_attributes(registry_train_df_shuffled)

# Transform
X_train_static_scaled_df = (
    pd.DataFrame(
        X_static_preprocessor.transform(X_train_static_df),
        columns=X_static_cols_names
    )
)


# _______________________
# Transform sets TEST set

# Merge
X_test_static_df = merge_attributes(registry_test_df)

# Transform
X_test_static_scaled_df = (
    pd.DataFrame(
        X_static_preprocessor.transform(X_test_static_df),
        columns=X_static_cols_names
    )
)

In [ ]:
if EXAMPLE_INPUT:
    display(X_train_static_scaled_df)
else:
    print(f"Shuffled registry df AFTER merging dimensions: {X_test_static_scaled_df.shape}")

# Save

## Reset train registry index 🚩

In [ ]:
# Train registry
(
    registry_train_df_shuffled
        .reset_index(
            inplace=True,
            drop=True
        )
)

display(registry_train_df_shuffled)

## Create the dictionary to collect data

In [ ]:
if not EXAMPLE_INPUT:
    model_feed = {
        
        # Split parameters
        'train_size': train_size,
        'test_size': test_size,
        'min_required_obs': min_required_obs,

        # Time series columns references        
        'X_minmax_columns_index': X_minmax_columns_index,
        'X_minmax_columns_names_list': X_minmax_columns_names_list,
        
        'X_standard_columns_index': X_standard_columns_index,
        'X_standard_columns_names_list': X_standard_columns_names_list,
        
        'X_no_scaled_columns_index': X_no_scaled_columns_index,
        'X_no_scaled_columns_names_list': X_no_scaled_columns_names_list,

        "X_cols_names": X_cols_names,

        # Attribute columns references
        'X_static_minmax_columns_index': X_static_minmax_columns_index,
        'X_static_minmax_columns_names_list': X_static_minmax_columns_names_list,
        
        'X_static_standard_columns_index': X_static_standard_columns_index,
        'X_static_standard_columns_names_list': X_static_standard_columns_names_list,
        
        'X_static_no_scaled_columns_index': X_static_no_scaled_columns_index,
        'X_static_no_scaled_columns_names_list': X_static_no_scaled_columns_names_list,

        "X_static_cols_names": X_static_cols_names,


        # Time series datasets
        "X_train": X_train_scaled_shuffled,
        "y_train": y_train_shuffled,
        
        "X_test": X_test_scaled,
        "y_test": y_test,

        # Attributes datasets
        "X_train_static_df": X_train_static_scaled_df,
        "X_test_static_df": X_test_static_scaled_df,
        
        "X_train_registry_df": registry_train_df_shuffled,
        "X_test_registry_df": registry_test_df,

        # Preprocessors
        "X_preprocessor": X_preprocessor,
        "X_static_preprocessor": X_static_preprocessor,
    }

## Save the dictionary

In [ ]:
# Store
if VALIDATION:
    folder = f"validation_data/"
else:
    folder = ""
if not EXAMPLE_INPUT:
    with open(
        os.path.join(
            camels_gb_silver_dir,
            f"{folder}model_feed-w{sequence_length}-{label_field_feed}-{start_year}-cs_{cs}-train_size_{int(train_size*10)}.pkl"
        ),
        'wb'
    ) as f:
        pickle.dump(model_feed, f)